# M3 · Statistical evaluation and pooling comparison
All numbers below are read from **actual saved posterior fits** and the source-derived M2B panel. Only evaluation opens `ground_truth.json`. Holdout results condition on the pre-outcome marketing plan and market context; they are not unconditional future forecasts. The promotional decomposition is model-based and assumption-dependent, not experiment-validated causal lift.

In [ ]:
import json
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from mmm_lab.evaluate import evaluate_models, posterior_components
from mmm_lab.model import load_model_data
from mmm_lab.paths import ARTIFACTS_DIR

evaluation_path = ARTIFACTS_DIR / 'evaluation.json'
if not evaluation_path.exists():
    evaluate_models()
results = json.loads(evaluation_path.read_text())
data = load_model_data()
assert results['model_input_sha256'] == data.input_sha256
models = results['models']

## Convergence and held-out accuracy
R-hat near or below 1.01, adequate bulk effective sample size and zero unresolved divergences are desired, but none alone proves model validity. WAPE is total absolute error divided by observed units; bias is predicted minus observed units. The 90% holdout interval includes Normal observation noise and is a **predictive** interval.

In [ ]:
rows = []
for name, model in models.items():
    rows.append({'pooling': name, **model['diagnostics'], **{f'train_{k}': v for k, v in model['train'].items() if not isinstance(v, dict)}, **{f'holdout_{k}': v for k, v in model['conditional_holdout'].items() if not isinstance(v, dict)}})
display(pd.DataFrame(rows).set_index('pooling'))
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar([r['pooling'] for r in rows], [r['holdout_wape'] for r in rows])
ax.set(ylabel='Conditional holdout WAPE', title='Predictive accuracy by pooling specification')
plt.show()

In [ ]:
manifest = json.loads((ARTIFACTS_DIR / 'fit_manifest.json').read_text())
partial = az.from_netcdf(ARTIFACTS_DIR / manifest['files']['partial']['path'])
baseline, channel, expected, noise = posterior_components(data, partial['posterior'])
segment = 'pacific_coast'
s = data.segments.index(segment)
mean = expected[:, s, :].mean(axis=0)
credible_low, credible_high = np.quantile(expected[:, s, :], [0.05, 0.95], axis=0)
rng = np.random.default_rng(47)
predictive = expected[:, s, :] + rng.normal(size=expected[:, s, :].shape) * noise[:, s, None]
predictive_low, predictive_high = np.quantile(predictive, [0.05, 0.95], axis=0)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(data.weeks, data.sales[s], label='Observed units', linewidth=1)
ax.plot(data.weeks, mean, label='Posterior expected units')
ax.fill_between(data.weeks, credible_low, credible_high, alpha=.18, label='90% expected-sales credible interval')
ax.fill_between(data.weeks, predictive_low, predictive_high, alpha=.13, label='90% future-observation predictive interval')
ax.axvline(data.weeks[100], color='black', linestyle='--', label='Holdout begins')
ax.set(title=f'{segment}: posterior predictive check', ylabel='Sales units')
ax.set_xticks([data.weeks[i] for i in (0, 25, 50, 75, 100, 119)])
ax.legend(ncol=2)
plt.show()

## Synthetic truth recovery and shrinkage
The coefficient comparison uses the generator's **maximum** channel effect, not the effect at observed spend. Its 90% intervals are parameter credible intervals. Contribution coverage compares each modeled segment-week/channel contribution with hidden synthetic truth. Recovery can fail even when held-out sales prediction is good because overlapping channels and baseline terms are confounded.

In [ ]:
display(pd.DataFrame([{'pooling': name, **{k:v for k,v in m['recovery'].items() if k not in ('coefficient_by_segment_channel','shared_shape_recovery')}} for name,m in models.items()]).set_index('pooling'))
display(pd.DataFrame(models['partial']['recovery']['shared_shape_recovery']))
comparison = pd.concat([pd.DataFrame(m['recovery']['coefficient_by_segment_channel']).assign(pooling=name) for name,m in models.items()])
display(comparison.query("segment_id in ['mountain_west','pacific_coast']").sort_values(['segment_id','channel','pooling']))
width = comparison.assign(interval_width=lambda d: d.credible_90_high-d.credible_90_low).pivot_table(index=['segment_id','channel'], columns='pooling', values='interval_width')
display(width)

## Correlated campaigns and limitations
The table below is computed on training planned spend. High pairwise correlations make individual channel response harder to identify; compare recovery and interval widths before favoring any pooling strategy. Partial pooling can reduce variance for weak-information regions and can bias heterogeneous true effects toward the population. Prior predictive checks in notebook 02 expose whether the declared priors generate plausible sales scales.

In [ ]:
display(pd.DataFrame(results['training_campaign_spend_correlations']).T)
display(pd.DataFrame([models['partial']['prior_sensitivity']]))
for limitation in results['limitations']:
    print('•', limitation)